In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import numpy as np

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# 1. Chuẩn bị dữ liệu cho mô hình

## 1.1. Lấy dữ liệu

- Đọc từ file CSV
- Đọc từ file ảnh
- Đối với dữ liệu dùng để học, đọc từ thư viện

In [4]:
data = fetch_california_housing()

In [5]:
data.data

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]])

In [6]:
data.data.shape

(20640, 8)

In [7]:
data.target

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894])

In [8]:
data.target.shape

(20640,)

## 1.2. Chia dữ liệu thành bộ train và bộ test

Thông thường, từ bộ dữ liệu ban đầu, ta chia thành 2 bộ train và test theo tỷ lệ 80-20 hoặc 70-30.

In [9]:
X = data.data
y = data.target

In [10]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
X_train.shape

(16512, 8)

In [12]:
X_test.shape

(4128, 8)

In [13]:
y_train.shape

(16512,)

In [14]:
y_test.shape

(4128,)

In [15]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# 2. Xây dựng mô hình

In [16]:
class MySimpleNet(nn.Module):
    def __init__(self): # bắt buộc phải có method này
        super(MySimpleNet, self).__init__()

        # 8 - số lượng đặc trưng đầu vào (8 đặc trưng của mỗi ngôi nhà)
        # 32 - số lượng nơ ron trong mạng
        # 1 - số lượng giá trị đầu ra (1 giá trị đầu ra của ngôi nhà)
        self.linear_1 = nn.Linear(8, 32)
        self.relu = nn.ReLU()

        self.linear_2 = nn.Linear(32, 1)

    def forward(self, x): # bắt buộc phải có method này
        x_1 = self.linear_1(x)
        x_2 = self.relu(x_1)
        y = self.linear_2(x_2)
        return y # output của mạng nơ ron

In [17]:
model = MySimpleNet()

In [18]:
model

MySimpleNet(
  (linear_1): Linear(in_features=8, out_features=32, bias=True)
  (relu): ReLU()
  (linear_2): Linear(in_features=32, out_features=1, bias=True)
)

# 3. Huấn luyện mô hình

## 3.1. Khởi tạo hàm Loss và thuật toán tối ưu Optimizer

In [19]:
loss_func = nn.MSELoss()
loss_func

MSELoss()

In [20]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)

## 3.2. Huấn luyện mô hình

In [21]:
num_epoch = 500
num_epoch_log = 50

In [22]:
device_str = 'cpu'
# device_str = 'cuda'

device = torch.device(device_str)

In [23]:
model = model.to(device)

In [24]:
for epoch in range(num_epoch):
    model.train()

    # Bước 1: Optimizer zero grad
    optimizer.zero_grad()

    # Bước 2: Foward data to model
    X_train = X_train.to(device)
    y_pred = model(X_train)

    # Bước 3: Tính giá trị loss
    y_train = y_train.to(device)
    loss_value = loss_func(y_pred, y_train)

    # Bước 4: Cập nhật trọng số của mô hình
    loss_value.backward()
    optimizer.step()

    # Bước 5: (Tuỳ chọn) In các thông số ra ngoài màn hình
    if epoch % 50 == 0 or epoch == num_epoch - 1:
        print(f'Epoch={epoch}', f'Training loss={loss_value.item()}')

/Users/minhhuunguyen/REPOSITORY/my_env/lib/python3.9/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([16512])) that is different to the input size (torch.Size([16512, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch=0 Training loss=23.02137565612793
Epoch=50 Training loss=8.539664268493652
Epoch=100 Training loss=2.1291067600250244
Epoch=150 Training loss=1.790182113647461
Epoch=200 Training loss=1.5887610912322998
Epoch=250 Training loss=1.4770328998565674
Epoch=300 Training loss=1.4177424907684326
Epoch=350 Training loss=1.38539719581604
Epoch=400 Training loss=1.3668960332870483
Epoch=450 Training loss=1.356068730354309
Epoch=499 Training loss=1.3496967554092407


# 4. Đánh giá mô hình

In [25]:
model.eval()

MySimpleNet(
  (linear_1): Linear(in_features=8, out_features=32, bias=True)
  (relu): ReLU()
  (linear_2): Linear(in_features=32, out_features=1, bias=True)
)

In [26]:
with torch.no_grad():
    X_test = X_test.to(device)
    y_pred_test = model(X_test)

    y_test = y_test.to(device)
    test_loss = loss_func(y_pred_test, y_test)
    print(f'Test loss {test_loss.item()}')

    if device_str == 'cuda':
        y_test = y_test.cpu()
        y_pred_test = y_pred_test.cpu()

Test loss 1.3230376243591309


/Users/minhhuunguyen/REPOSITORY/my_env/lib/python3.9/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([4128])) that is different to the input size (torch.Size([4128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [27]:
model.train()

MySimpleNet(
  (linear_1): Linear(in_features=8, out_features=32, bias=True)
  (relu): ReLU()
  (linear_2): Linear(in_features=32, out_features=1, bias=True)
)